In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score,f1_score
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv')
df.head(5)

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


In [3]:
# data preprocessing



def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)


def normalize_text(df):
    """Normalize the text data."""
    try:
        # Use 'content' instead of 'text' since that's the actual column name
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        raise
  

In [4]:
df = normalize_text(df)
df.head(5)

,tweet_id,sentiment,content
0,1956967341,empty,tiffanylue know listenin bad habit earlier sta...
1,1956967666,sadness,layin n bed headache ughhhh waitin call
2,1956967696,sadness,funeral ceremony gloomy friday
3,1956967789,enthusiasm,want hang friend soon
4,1956968416,neutral,dannycastillo want trade someone houston ticke...


In [5]:
X = df['sentiment'].isin(['happiness' , 'sadness'])
df = df[X]

In [6]:
df['sentiment'] = df['sentiment'].replace({'happiness':1 , 'sadness':0})

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_16852\3399834364.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'happiness':1 , 'sadness':0})


In [7]:
df.sample(5)

,tweet_id,sentiment,content
17968,1965850031,0,dark knight dvd missing pissed
1,1956967666,0,layin n bed headache ughhhh waitin call
4397,1960345960,0,field day sad ribbon me
9864,1962730313,0,feeling well food poisoning
15749,1964974000,1,wow really need fun tonight


In [8]:
vectorizer = CountVectorizer(max_features = 1000)
X = vectorizer.fit_transform(df['content'])
y = df['sentiment']

In [9]:
X_train , X_test , y_train, y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [12]:
import dagshub
dagshub.init(repo_owner='Aayush10671', repo_name='emotion-detection', mlflow=True)

mlflow.set_tracking_uri('https://dagshub.com/Aayush10671/emotion-detection.mlflow')
mlflow.set_experiment("LogisticRegression model")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=318ae033-8b2f-4c71-9790-a92f89c409bd&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4b29c4be8f30058230325e03c70358db048477b49a2dc10139bebc4a518e2028




Accessing as Aayush10671

Initialized MLflow to track repo "Aayush10671/emotion-detection"

Repository Aayush10671/emotion-detection initialized!

2026/07/17 10:24:53 INFO mlflow.tracking.fluent: Experiment with name 'LogisticRegression model' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/0d8785b61c4f4b939f6c49ac86a37527', creation_time=1784264093751, experiment_id='0', last_update_time=1784264093751, lifecycle_stage='active', name='LogisticRegression model', tags={}, workspace='default'>

In [13]:
with mlflow.start_run():
    mlflow.log_param("vectorizer" , "bag of words")
    mlflow.log_param("max_features" , 1000)
    mlflow.log_param("test_size" , 0.2)

    model = LogisticRegression()
    model.fit(X_train , y_train)

    mlflow.log_param("model" , "LogisticRegression")

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_pred , y_test)
    precision = precision_score(y_pred , y_test)
    recall = recall_score(y_pred , y_test)
    f1 = f1_score(y_pred , y_test)

    mlflow.log_metric("accuracy" , accuracy)
    mlflow.log_metric("precision" , precision)
    mlflow.log_metric("recall" , recall)
    mlflow.log_metric("f1-score" , f1)
    mlflow.sklearn.log_model(model , "model")


    import os
    notebook_path = 'exp1.ipynb'
    os.system(f"jupyter nbconvert --to notebook --execute --inplace  {notebook_path}")
    mlflow.log_artifact(notebook_path)


    print("accuracy = " , accuracy)
    print("recall = " , recall)
    print("precision = ", precision)
    print("f1-score = ", f1)


2026/07/17 10:34:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/17 10:34:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


accuracy =  0.776867469879518
recall =  0.7690058479532164
precision =  0.7773399014778325
f1-score =  0.7731504164625184
🏃 View run victorious-owl-239 at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/0/runs/6fb69a7bcce44fdbaea7806f1837742e
🧪 View experiment at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/0
